# Suite QRY — Queries

A predicate is a query: `select(predicate, variables)` streams its matches among a store's data (each symbol's extent:
what the store's singletons reach) for which its rule holds. `Queries.Scan(store)` makes any store queryable in memory,
planning each query from the rule's shape; `explain` shows the plan.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Constraints as C, Queries as Q } from "@mbse/patterns";
import { Bindings as Bd, Errors, Schemas as S } from "@mbse/schemas/Framework";
import { Contact, Phone, Phones, assert, book, equal, listed, native, protocol_problems, raises, same } from "./support.js";

const [the, c, p] = [E.variable("the"), E.variable("c"), E.variable("p")];

function predicate(rule: unknown, symbols: Record<string, S.OfObject.Data> = { the: Contact }): C.OfPredicate.Data {
  return new C.OfPredicate.Builder().name("Q").symbols(symbols).rule(rule).create();
}

const store = book();
const Pager = new S.OfObject.Builder().name("Pager").ref().properties(native("number")).relations(
  (r: any) => r.name("owners").of(Phones).me("phone")).create(); // a pager may fill a phone's link too
store.register(Pager);
const ann = store.Contact().name("ann").age(30n).phones((e: any) => e.phone(store.Phone().number("1").create())).phones(
  (e: any) => e.phone(store.Phone().number("2").create())).phones((e: any) => e.phone(store.Pager().number("p").create())).create();
const bob = store.Contact().name("bob").phones((e: any) => e.phone(store.Phone().create())).create();
const cy = store.Contact().name("cy").age(9n).create();
listed(store, ann, bob, cy);
const loose = store.Contact().name("dee").age(50n).create(); // never listed: not the store's data
const query = new Q.Scan(store);

function names(matches: Iterable<Q.Match>): Record<string, unknown>[] {
  return [...matches].map((match) => Object.fromEntries(Object.entries(match).map(([symbol, value]: [string, any]) =>
    [symbol, value.schema_name() === "Contact" ? value.name : value.number])));
}

## QRY-01 · A query streams the matches for which its rule holds

In [ ]:
{
  const adults = query.select(predicate(the.age.ge(18n)));
  assert(adults[Symbol.iterator]() === adults && equal(adults.next().value, { the: ann })); // an iterator of matches
  assert([...adults].length === 0); // the rest: bob's age is unknown, cy is a child, dee is not in the extent
  assert(equal([...query.select(predicate(the.age.ge(18n)), null, true)], [{ the: ann }, { the: bob }]));
  assert(equal(names(query.select(predicate(the.has("name")))), [{ the: "ann" }, { the: "bob" }, { the: "cy" }]));
}

## QRY-02 · Variables are bound once, and objects too

In [ ]:
{
  assert(equal([...query.select(predicate(the.age.ge(E.variable("min"))), { min: 21n })], [{ the: ann }]));
  assert(equal([...query.select(predicate(the.age.lt(E.variable("max")).and_(the.ne(E.variable("not_")))),
    { max: 40n, not_: cy })], [{ the: ann }])); // an object has no literal: bound per match
  assert([...query.select(predicate(E.variable("flag").or_(the.age.ge(99n))), { flag: true })].length === 3);
  assert(same(query.explain(predicate(E.variable("flag").and_(the.has("name"))), { flag: false }), [
    "first 1 test", "the: scan Contact"])); // the variables decide the rule: false, tested before any symbol is bound
  assert([...query.select(predicate(E.variable("flag").and_(the.has("name"))), { flag: false })].length === 0);
  assert(same(query.explain(predicate(E.variable("go").and_(the.has("name"))), { go: E.variable("x") }), [
    "first 1 test", "the: scan Contact, then 1 test"])); // a variable bound to no literal is tested before any symbol
}

## QRY-03 · The extents are read when the first match is asked for

In [ ]:
{
  const pending = query.select(predicate(the.age.ge(40n)));
  listed(store, loose); // listed after the query was made, before it was read
  assert(equal([...pending], [{ the: loose }]));
}

## QRY-04 · A query that cannot be right is refused when it is made

In [ ]:
{
  raises(Errors.ValueError, () => query.select(new C.OfPredicate.Builder().rule(true).create()),
    "a query needs a predicate with at least one symbol");
  raises(Errors.ValueError, () => query.select(predicate(true, { v: new S.OfObject.Builder().name("V").create() })),
    "symbol 'v' needs a named reference object schema");
  raises(Errors.AttributeError, () => query.select(predicate(true, { x: new S.OfObject.Builder().name("Elsewhere").ref().create() })),
    "no schema registered as 'Elsewhere'");
  raises(Errors.ValueError, () => query.select(predicate(the.has("name")), { the: ann }), "'the' is a symbol; it is not a variable");
  raises(Errors.ValueError, () => query.select(predicate(the.age.ge(E.variable("n")))), "the predicate cannot be a query: ");
  raises(Errors.ValueError, () => query.select(predicate(E.operation("pow", the.age, 2n).ge(1n))), "'pow' is not a core operation");
  const numbers = query.select(predicate(the.age)); // a rule must give a bool, found as matches are read
  raises(TypeError, () => [...numbers], "a predicate must be a bool, got int");
}

## QRY-05 · Several symbols match the cross product, narrowed by hops through relations

In [ ]:
{
  const anyPhone = E.quantifier("any", "e", E.operation("entries", c, "phones"), E.variable("e").phone.eq(p));
  const owned = predicate(anyPhone.and_(p.has("number")), { c: Contact, p: Phone });
  assert(same(query.explain(owned), ["c: scan Contact", "p: c.phones to phone, any number, then 2 tests"]));
  assert(equal(names(query.select(owned)), [{ c: "ann", p: "1" }, { c: "ann", p: "2" }])); // ann's pager is not a phone
  const anyOwner = E.quantifier("any", "e", E.operation("entries", p, "owners"), E.variable("e").owner.eq(c));
  const byPhone = predicate(anyOwner.and_(c.age.ge(18n)), { c: Contact, p: Phone });
  assert(same(query.explain(byPhone), ["p: scan Phone", "c: p.owners to owner, at most one, then 2 tests"])); // unique(owner)
  assert(equal(names(query.select(byPhone)), [{ c: "ann", p: "1" }, { c: "ann", p: "2" }]));
  const pairs = predicate(c.age.gt(the.age), { the: Contact, c: Contact }); // no hop: the cross product, tested once both are bound
  assert(same(query.explain(pairs), ["the: scan Contact", "c: scan Contact, then 1 test"]));
  assert(equal(names(query.select(pairs)), [{ the: "ann", c: "dee" }, { the: "cy", c: "ann" }, { the: "cy", c: "dee" }]));
  const both = predicate(anyPhone.and_(anyOwner).and_(p.has("number")), { c: Contact, p: Phone }); // hops both ways: either first
  assert(same(query.explain(both), ["c: scan Contact", "p: c.phones to phone, any number, then 3 tests"]));
  assert(equal(names(query.select(both)), [{ c: "ann", p: "1" }, { c: "ann", p: "2" }]));
  const narrow = predicate(the.has("name").and_(anyPhone).and_(c.ne(the)), { the: Contact, c: Contact, p: Phone });
  assert(same(query.explain(narrow), ["the: scan Contact, then 1 test", "c: scan Contact, then 1 test",
    "p: c.phones to phone, any number, then 1 test"]));
  assert([...query.select(narrow)].length === 9); // three named contacts, each with ann's two phones and bob's one, but itself
}

## QRY-06 · Only a hop through a relation narrows candidates

In [ ]:
{
  const plan = (rule: unknown) => query.explain(predicate(rule, { c: Contact, p: Phone }));
  const e = E.variable("e");
  const scans = ["c: scan Contact", "p: scan Phone, then 1 test"];
  assert(same(plan(E.quantifier("all", "e", E.operation("entries", c, "phones"), e.phone.eq(p))), scans)); // not any
  assert(same(plan(E.quantifier("any", "e", E.operation("count", c), e.phone.eq(p))), scans)); // not entries
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.phone.ne(p))), scans)); // not eq
  assert(same(query.explain(predicate(E.quantifier("any", "e", E.operation("entries", E.variable("x"), "phones"), e.phone.eq(p)),
    { c: Contact, p: Phone }), { x: ann }), scans)); // from a variable, not a symbol
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, "nope"), e.phone.eq(p))), scans)); // no such adjacency
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, 1n), e.phone.eq(p))), scans)); // not an adjacency's name
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.label.eq(p))), scans)); // a property
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.owner.eq(p))), scans)); // its own link
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.phone.eq(c))), [
    "c: scan Contact, then 1 test", "p: scan Phone"])); // back to itself: a test of c alone
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), e.eq(p))), scans)); // not a link's target
  assert(same(plan(E.quantifier("any", "e", E.operation("entries", c, "phones"), p.eq(e.phone))), [
    "c: scan Contact", "p: c.phones to phone, any number, then 1 test"])); // either side of eq
}

## QRY-07 · Scan is a store, and select asks a queryable store or scans another

In [ ]:
{
  assert(protocol_problems(query).length === 0 && same(protocol_problems(store), ["select"]));
  assert(query.schema("Contact") === Contact && query.registered("Listed") === store.registered("Listed")
    && query.name_of(Contact) === "Contact");
  assert(same(query.names(), store.names()) && same(query.extent("Contact"), store.extent("Contact")));
  assert(query.singleton("book.Directory") === store.singleton("book.Directory"));
  assert(query.member(ann, "name") === "ann" && query.builder("Contact").name("e").create().name === "e");
  assert(query.builder("Contact", ann).update() === ann && query.store === store);
  const scanned = Q.select(store, predicate(the.age.ge(18n))); // a plain store is scanned; the update above reordered its entries
  assert(same([...scanned].map((m: any) => m.the.name).sort(), ["ann", "dee"]));

  /** A store that answers queries itself. */
  class Native {
    select(predicate: C.OfPredicate.Data, variables: unknown = null, unknown = false) {
      return [["asked", predicate.name, variables, unknown]][Symbol.iterator]();
    }
  }

  const [asked] = [...Q.select(new Native() as any, predicate(true), { v: 1n }, true)] as any[];
  assert(asked[0] === "asked" && asked[1] === "Q" && asked[2].v === 1n && asked[3] === true);
  const Item = new S.OfObject.Builder().name("Item").ref().singleton("the.Item").properties(native("name")).create();

  class Thing {
    constructor(public name: string | null) {}

    identity(): unknown {
      return "thing";
    }

    schema_name(): string {
      return "Item";
    }

    owner(): null {
      return null;
    }

    accept(visitor: any): void {
      Bd.accept(binding, this, visitor);
    }
  }

  const binding: Bd.Binding = new Bd.Binding(Item, (t: Thing) => new Bd.State(new Map([["name", t.name]])),
    (s) => new Thing((s.values.get("name") as string | undefined) ?? null));
  const bound = new Bd.OfStore([[Item, (instance?: Thing) => new Bd.Builder(binding, instance)]]);
  const things = Q.select(bound, predicate(the.has("name").not_(), { the: Item }));
  assert(equal([...things], [{ the: bound.singleton("the.Item") }])); // bound classes too
}